# Capstone: your competition agent

**You have one file: tune it once more, test it fairly, then show what every stage of it bought.**

The race is one track, CARLA's Monza, three laps, raced by one file.
Thirteen units built that file a stage at a time: Unit 1's starter, Unit 4's PID, Unit 5's pure pursuit, Unit 6's corner cap, Unit 7's speed profile, Unit 8's measured grip, Unit 9's racing line, Unit 10's MPC, Unit 11's respawn fallback and margin, Unit 12's crash-aware search, and Unit 13's race-day re-tune on `monza-like`.
This project asks two questions about that file, and wants the evidence for both.

1. **Can you still make it better, honestly?** Make **one** choice of your own, by Unit 13's protocol, and test it on cars you have never raced, once, by a rule you write down before the race.
2. **What did every stage buy?** Build the file back up from the starter, one unit's idea at a time, and measure what each stage bought in race time, with its spread, and what it bought that race time does not show.

**The bar.** Unit 1's starter is the bar every stage is measured against.
On the car you race it does not even finish: Milestone 1 makes it clean with its own one knob, and that clean starter is row 0 of your table.
Your submitted file has to beat it by Unit 1's rule, at 0 collisions, on race seeds 0 to 4.

**Three files**, named in every milestone:

- the **incumbent**: the file you start from, `agents/unit-13.py` (or your own race-day file from Unit 13, if it keeps the book's mode switches: `STEERING`, `SPEED_MODE`, `MU_G`, `LINE_CONTROL`, `RESPAWN_IMPULSE`);
- the **candidate**: the incumbent with your one choice, frozen before its test;
- the **submitted file**: the candidate if it passes its test, otherwise the incumbent.

**The rules of the build.**

- Every race is on `monza-like` (racekit's model of Monza), 3 laps, on **`carla_like(noise=True)`**, the book's CARLA stand-in since Unit 6, with **the timing off**, through the setup's `race()` helper.
  The one timed race is Milestone 4's `check_submission`.
- A race is **clean** when it finishes with 0 collisions and never more than 2 m from its own line (the centreline for a file with no line).
- Cars: the nominal car at race seeds 0 to 4; the **training** cars 16, 18 and 8 and the **validation** cars 24 to 47 of Unit 11's perturbed family (car k races at race seed k mod 5); and **your own test cars 120 to 143**, raced once, in Milestone 2's test cell, and never again.
- **Never race** cars 72 to 95 (Unit 13's test cars) or 96 to 119 (plan `roar-014`'s), cars k ≥ 1000 (sealed), or the tracks `test-circuit-seed2027` and `test-circuit-seed2028`: a test set raced to make a choice is spent.
  The setup's `race()` refuses the cars.
- Every number in this project is a number about racekit's carla-like car: a model, not CARLA's car.

**The races.**
The bar about 10, your choice about 30 on the training objective plus 24 validation and 24 test, the build-up 55, the leave-one-out 20, and qualifying about 10: **about 175 races, about 24 CPU-minutes** if run one after another.
Spread them over the three sittings and at home.
Milestones 1 and 2 need your decisions between races; Milestone 3's two long cells (the build-up and the leave-one-out) need none once the submitted file is fixed, so start each one and write Milestone 2's evidence and Milestone 5's first answers while it runs.
If your time is short, you may run the build-up and the leave-one-out on race seeds 0 to 2 instead of 0 to 4: say so in the report, next to every number it touches.

Run the setup cell first.

## Setup

The cell loads the track and the incumbent, and gives you the helpers of Unit 13's exercises.
`race(source, seed=0, k=None, laps=3, cap=900.0, test=False, **constants)` races a copy of `source` with `constants` changed (each written with `repr`, Unit 12) and the timing off (the starter and `unit-07.py` have none to switch off), on `monza-like`, on `carla_like(noise=True)`, cut at `cap` seconds; with `k` it races car k of the perturbed family at race seed k mod 5 (the `seed` argument is then ignored).
It refuses cars 72 to 143 and k ≥ 1000, except that `test=True` lets it race each of **your** test cars 120 to 143 once in this session: only Milestone 2's test cell passes `test=True`.
The refusal is a seat belt, not a record: the record is the pre-registration you write in Milestone 2.

`LINE` is the incumbent's racing line on `monza-like` and `CENTRELINE` the line of a file with `LINE_CONTROL = None`; pass the right one to `clean()` and `line_error()`.
`incumbent` holds the incumbent's module constants (`incumbent.TARGET_SPEED`, and so on).

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENTS = Path("../racekit/agents")
STARTER, UNIT_07, UNIT_13 = AGENTS / "starter.py", AGENTS / "unit-07.py", AGENTS / "unit-13.py"

# The three files of this project.
INCUMBENT = UNIT_13  # the file you start from (or your own race-day file from Unit 13, with the book's mode switches)
CANDIDATE = None  # Milestone 2: the incumbent with your one choice, frozen before its test
SUBMITTED = None  # Milestone 2: the candidate if it passes its test, otherwise the incumbent

workdir = Path(mkdtemp(prefix="roar-capstone-"))
TRACK = "monza-like"
TICK = 0.05 + 1e-9  # s: one control tick (race times are multiples of it)
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
CAP = 900.0  # s: a race is cut here unless a cap is given
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race
UNTIMED_FILES = (STARTER, UNIT_07)  # written before Unit 10: no timing inside them to switch off
SEEDS = (0, 1, 2, 3, 4)  # the nominal car's race seeds
TRAINING_CARS = (16, 18, 8)  # Unit 13's objective: the three lowest-grip training cars
VALIDATION_CARS = range(24, 48)
TEST_CARS = range(120, 144)  # YOUR test cars: raced once, by Milestone 2's test cell only
# Cars 72-95 (Unit 13's test cars) and 96-119 (plan roar-014's) are spent; cars k >= 1000 are sealed.

track = load_track(TRACK)
with shim_installed():
    Incumbent, incumbent_name = load_submission(INCUMBENT)
incumbent = sys.modules[incumbent_name]  # the incumbent's module constants
LINE = np.array(Incumbent(track.waypoints, None).offsets)  # m: the incumbent's racing line, as offsets
CENTRELINE = np.zeros(len(track))  # m: the line of a file that races the centreline
written = []  # every variant file written so far
tested = []  # the test cars raced in this session


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def race(source=INCUMBENT, seed=0, k=None, laps=3, cap=CAP, test=False, **constants):
    # One race on monza-like of `source` with `constants` changed and the timing off, on the carla-like car with
    # its noise (car k of Unit 11's perturbed family if k is given, at race seed k % 5).
    # The timing off is the only change a race makes to a file: a test race of CANDIDATE is CANDIDATE, untimed.
    assert not test or (k is not None and k in TEST_CARS), "test=True is only for your test cars 120-143"
    if k is not None:
        assert k < 72 or (test and k in TEST_CARS), "cars 72-119 are spent and k >= 1000 sealed; 120-143 are tests"
        assert k not in tested, f"car {k} has had its one test race"
        if test:
            tested.append(k)
        seed = k % 5
    if source in UNTIMED_FILES:
        path = variant(source, **constants)
    else:
        path = variant(source, **constants, **NO_TIMING)
    return run_race(path, TRACK, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=cap)


def error_trace(result, offsets=LINE):
    # Unit 9: the time and the cross-track error from the race's own line over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return telemetry["t"][driving], telemetry["offset"][driving] - target


def line_error(result, offsets=LINE):
    return float(np.max(np.abs(error_trace(result, offsets)[1])))


def clean(result, offsets=LINE):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def in_ticks(times):
    # Unit 12's habit: race times (and sums of them) as whole ticks before any comparison.
    return [round(t / DT) for t in times]


print(f"incumbent {INCUMBENT.name}: TARGET_SPEED {incumbent.TARGET_SPEED}, STEERING {incumbent.STEERING!r}, "
      f"{len(LINE)} line offsets")

## Milestone 1

### The bar

Unit 1 raced `agents/starter.py` (the same file as `unit-01.py`) on `monza-like` on `KinematicCar`, and it was clean: about 870 s, Unit 1's own measurement.
Race the same file on the same track here, on the car you race.

1. Race the starter as shipped on `monza-like`, race seed 0, cut at 120 s: `race(STARTER, cap=120.0)` (no constants: the starter has no timing).
   Print its time, whether it finished, its collisions and its largest error from the centreline.
2. In `why_not_clean`, two sentences: Unit 1 measured this file clean; what changed, and why does that make the starter crash where it did not before?
   (Unit 6's hook raced the same file on the grip car.)
3. Make the starter clean with its **own one knob**, `TARGET_SPEED` (it ships 20.0): race it at 10, 12, 14 and 16 m/s at race seed 0, cut at 2,400 s, and record for each the time, the collisions, the largest error from the centreline and whether it is clean.
4. `bar_speed`: the **fastest clean** grid speed.
   Race it at race seeds 0 to 4 (seed 0 is already raced): `bar_times`, `bar_clean`.
   That five-race set is **the bar**, row S0 of your ablation.

In [ ]:
shipped_starter = None  # TODO: race(STARTER, cap=120.0)
# TODO: print its time, finished, collisions and line_error(shipped_starter, CENTRELINE)

why_not_clean = "TODO"

grid = {}  # TODO: {speed: result} for TARGET_SPEED 10.0, 12.0, 14.0, 16.0 at seed 0, cap=2400.0
grid_table = []  # TODO: one row per speed: (speed, time, collisions, error from the centreline, clean)

bar_speed = None  # TODO: the fastest clean speed of the grid
bar_results = []  # TODO: the starter at bar_speed on each of SEEDS (reuse the seed-0 race)
bar_times = None  # TODO: the five times (s)
bar_clean = None  # TODO: how many of the five are clean (against CENTRELINE)

## Milestone 2

### Your choice, and its test

Tune **one** knob the incumbent has never been tuned on, by Unit 13's protocol, then test it once.

**The knob.** Choose one of `TARGET_SPEED` (Unit 4's top speed), `PROFILE_LEAD_S` (Unit 7's lead), `MPC_Q_E` or `MPC_R_DU` (Unit 10's weights), or `CURVATURE_HORIZON` (Unit 6's look-ahead).
Write down in `knob_reason` why that one: which unit set it, on what car or track, and why it may no longer be right on race day.
Pick a grid step for it and write it down **before** any race (the solution's `TARGET_SPEED` used 2.5 m/s).
If you choose `TARGET_SPEED`, the knob the solution chose, also run a **second** knob through training and validation (steps 1 and 2 below) and report its evidence: the test cars test one file only, your candidate, so the second knob's evidence stops at validation.

**1. Training: Unit 13's crash-aware objective.**
A setting is raced on the nominal car at race seeds 0 and 1 and on the training cars 16, 18 and 8 (five races).
Its two **per-replicate sums** are the nominal race at seed r plus the three cars' times.
A setting is **not clean** if any of its five races is not clean (it is rejected: it counts as infinitely slow).
A clean setting **beats** another when `noise_aware_better(in_ticks(its sums), in_ticks(the other's sums))`: Unit 1's rule on whole ticks (Unit 12's floating-point trap).
From the shipped value, step one grid step at a time in the faster direction while each step beats the last; the first value that is not clean ends the walk.
The **edge** is the highest clean value; Unit 11's margin rule ships **one grid step inside the edge**.
A knob that never reaches a value that is not clean has no edge and gets no margin: keep the last value that beat.

**2. Validation.**
Race your margin value on the 24 validation cars 24 to 47.
It must be clean on **all 24** (Unit 13's rule: N of N on every set of cars it was not tuned on).
If it is not, keep the shipped value: there is no candidate, no test race and no candidate row in Milestone 3, and your report gives the evidence.
The validation cars judge a value already found; if they change your choice, they are spent.

**3. Freeze.**
Write the candidate as a real file in a folder of your own **outside this repository** (the `roar-runs` folder the labs use): `CANDIDATE = write_variant(INCUMBENT, RUNS / "candidate.py", {knob: repr(value)})`.
> This repository is public. Keep your candidate, your records and your report on your own machine, never in a commit.

Edit its header comment **now** (what changed, by which search, and why: Unit 12's packaging), then leave the file alone: the file you test is the file you submit.
The test races it, like every race in this notebook, with only its timing switched off (`NO_TIMING`, so no wall clock decides a race); Milestone 4's `check_submission` races it exactly as submitted, timing on.

**4. The test, stated before any race of it.**
In the pre-registration cell, write down the frozen file, its one change, the cars (120 to 143, race seed k mod 5, 3 laps), the rule (**clean on every one of the 24**) and the decision (submit the candidate if it passes, otherwise the incumbent).
Run that cell, then the test cell: it races the candidate once on each of your test cars, with `test=True`.
Record the result as it comes.
**Never revise the choice on the test**: a test can veto a choice; it never selects a new one.

**5. The submitted file.**
`SUBMITTED` is `CANDIDATE` if the test passed, otherwise `INCUMBENT`.
`INCUMBENT`'s own test evidence is Unit 13's: clean on all 24 of its test cars 72 to 95 (or your own Unit 13 record, for your own file).

`WORKED_CHOICE` is the solution's own attempt at this milestone, for comparison after you have made yours.

In [ ]:
# The probe record of plan roar-014 (p8, p8v, p10): the solution's worked choice, from agents/unit-13.py.
# TARGET_SPEED (shipped 45.0, grid step 2.5): 47.5, 50.0 and 52.5 beat on the training objective, 55.0 was not
# clean (car 8: 1 collision), so the edge is 52.5 and the margin value 50.0. Validation cars 24-47: 50.0 clean on
# 24 of 24 (52.5, without the margin, 21 of 24). Test cars 96-119, pre-registered and raced once: 23 of 24 clean
# (car 117, race seed 2: 2 collisions, 590.05 s, max line error 4.485 m). By the rule, not submitted.
WORKED_CHOICE = {
    "knob": "TARGET_SPEED", "value": 50.0, "nominal seeds 0, 1 (s)": (447.80, 447.90),
    "validation clean": (24, 24), "test clean": (23, 24), "submitted": False,
}

In [ ]:
knob = "TODO"  # one of "TARGET_SPEED", "PROFILE_LEAD_S", "MPC_Q_E", "MPC_R_DU", "CURVATURE_HORIZON"
knob_reason = "TODO"
grid_step = None  # TODO: written down before any race


def objective(**constants):
    # TODO: race the incumbent with `constants` at seeds 0 and 1 and on TRAINING_CARS (five races);
    # return (the two per-replicate sums, a dict {name of each race that was not clean: (collisions, line error)})
    return None, None


training = {}  # TODO: {value: objective(**{knob: value})} for every value you raced, in order, starting with the shipped one
accepted = []  # TODO: the values that beat the previous one, in order
edge = None  # TODO: the highest clean value (None if no value was not clean)
margin_value = None  # TODO: one grid step inside the edge (or the last value that beat, if there is no edge)

validation = {}  # TODO: {k: result} for k in VALIDATION_CARS, the incumbent with {knob: margin_value}
validation_clean = None  # TODO: how many of the 24 are clean (against LINE, or CENTRELINE if your knob removes the line)
has_candidate = None  # TODO: True when validation_clean == 24

second_knob_evidence = "TODO"  # only if knob == "TARGET_SPEED": the second knob's training and validation, in words

In [ ]:
# Freeze (only if has_candidate): write candidate.py OUTSIDE this repository (it is public), then edit its header.
RUNS = None  # TODO: Path(...) of a folder of your own outside the repository, e.g. your roar-runs folder
CANDIDATE = None  # TODO: write_variant(INCUMBENT, RUNS / "candidate.py", {knob: repr(margin_value)})

# The pre-registration: fill it in and run this cell BEFORE the test cell.
PRE_REGISTRATION = {
    "file": "TODO",  # the frozen candidate's path
    "change": "TODO",  # its one change from the incumbent
    "cars": "120-143, race seed k % 5, 3 laps, monza-like, carla_like(noise=True), timing off",
    "rule": "TODO",  # the pass/fail rule
    "decision": "TODO",  # what you submit if it passes, and if it fails
    "written": "TODO",  # the date and time
}
print(PRE_REGISTRATION)

In [ ]:
# The test: run once, after the pre-registration. Every car races the frozen candidate once (test=True).
test_results = {}  # TODO: {k: race(CANDIDATE, k=k, test=True)} for k in TEST_CARS
test_clean = None  # TODO: how many of the 24 are clean
test_failures = {}  # TODO: {k: (collisions, time, line error)} for each test car that was not clean

SUBMITTED = None  # TODO: CANDIDATE if it passed the pre-registered rule, otherwise INCUMBENT
print("submitted:", SUBMITTED)

## Milestone 3

### Build it up: what every stage bought

An **ablation** builds the submitted file from the bar, one unit's idea per stage, and races each stage on the same seeds.
`REFERENCE_GRID` (below) gives the constants of each stage for `agents/unit-13.py`, written over the incumbent: S1 to S8 run with the respawn fallback **off** (`RESPAWN_IMPULSE = 1e300`) and the knobs before Unit 13's re-tune (`MU_FRACTION` 0.54, `A_DEC` 5.0, `A_ACC` 4.0); a constant not listed keeps the incumbent's value.
Two stages need care:

- **S2a and S2b**: changing the steering and the speed in one stage would hide which bought what, so S2a changes only the steering (pursuit at S1's 14 m/s) and S2b only the speed (pursuit's next clean m/s: the record raced PID at 15 m/s and it crashed on every seed).
- **S3**: Unit 6's corner cap cannot run at 45 m/s on this track (its 50 m look-ahead cannot brake from about 39 m/s: Unit 7's motivation), so it runs at its own fastest clean `TARGET_SPEED`, 32.5.

If a stage's given constants are not clean for **your** incumbent, find that stage's own fastest clean value, as S2b and S3 did, and say so.
The last row is the candidate, **S10**, marked submitted or not; if you kept the shipped value, there is no S10.
The submitted file is S9 (the incumbent) or S10 (the candidate).

1. Race the bar (S0, from Milestone 1: no new races) and every stage at race seeds 0 to 4 with its `STAGE_CAP`; S0 to S5 race the centreline (`CENTRELINE`), S6 onwards the line (`LINE`).
2. Build `ablation`: one row per stage with the stage, its idea, how many of 5 are clean, the median and the spread (largest minus smallest) of the five times, the **time bought** (the previous stage's median minus this one's), **Unit 1's verdict against the previous stage** (`noise_aware_better(this stage's times, the previous stage's times)`), and the largest line error of the five races.
   Print it as a table.
3. Plot it (`fig`, `axes`): the five times and the median of every stage against the stage's position in `STAGE_ORDER`, and below it the largest line error with the 2 m clean limit drawn; axis labels, a title, and a legend.
4. **Leave one out.** From the submitted file, remove one thing at a time and race each at seeds 0 to 4: **no line** (`LINE_CONTROL = None`, scored against `CENTRELINE`), **no grip** (`MU_G = None`), **no MPC** (`STEERING = "pursuit"`), and **no tune** (the last tuning in the submitted file: your choice if the candidate was submitted, otherwise Unit 13's re-tune, `PRE_TUNE`).
   `leave_one_out`: one row per removal with its median, its cost (its median minus the submitted file's), Unit 1's verdict (`noise_aware_better(the submitted file's times, its times)`), and how many are clean.

In [ ]:
# The reference grid of plan roar-014: each stage's constants, written over the incumbent (agents/unit-13.py).
OFF = {"RESPAWN_IMPULSE": 1e300}  # the respawn fallback off: no impulse is ever this large
PRE_TUNE = {"MU_FRACTION": 0.54, "A_DEC": 5.0, "A_ACC": 4.0}  # the knobs before Unit 13's re-tune
NO_LINE_NO_GRIP = {"MU_G": None, "LINE_CONTROL": None}
REFERENCE_GRID = {
    "S1": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pid", "SPEED_MODE": "flat", "TARGET_SPEED": 14.0},
    "S2a": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "flat", "TARGET_SPEED": 14.0},
    "S2b": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "flat", "TARGET_SPEED": 15.0},
    "S3": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "cap", "A_LAT": 4.6,
           "TARGET_SPEED": 32.5},
    "S4": {**OFF, **PRE_TUNE, **NO_LINE_NO_GRIP, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6,
           "TARGET_SPEED": 45.0},
    "S5": {**OFF, **PRE_TUNE, "LINE_CONTROL": None, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6,
           "TARGET_SPEED": 45.0},
    "S6": {**OFF, **PRE_TUNE, "STEERING": "pursuit", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S7": {**OFF, **PRE_TUNE, "STEERING": "mpc", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S8": {**PRE_TUNE, "STEERING": "mpc", "SPEED_MODE": "profile", "A_LAT": 4.6, "TARGET_SPEED": 45.0},
    "S9": {},  # the incumbent as shipped: Unit 13's re-tune
}
STAGE_IDEA = {
    "S0": "the bar: starter.py at its fastest clean TARGET_SPEED",
    "S1": "Unit 4: PID steering, a flat 14 m/s, the centreline, no measured grip",
    "S2a": "Unit 5: pure pursuit, the same 14 m/s",
    "S2b": "Unit 5's speed: pursuit at 15 m/s",
    "S3": "Unit 6: the corner cap (A_LAT 4.6, TARGET_SPEED 32.5)",
    "S4": "Unit 7: the speed profile (TARGET_SPEED 45)",
    "S5": "Unit 8: the measured grip (MU_G, MU_FRACTION 0.54)",
    "S6": "Unit 9: the racing line",
    "S7": "Unit 10: MPC steering",
    "S8": "Unit 11: the respawn fallback on",
    "S9": "Unit 13: the race-day re-tune (the incumbent)",
    "S10": "your choice (the candidate)",
}
STAGE_CAP = {"S1": 2400.0, "S2a": 1800.0, "S2b": 1800.0, "S3": 1800.0}  # s; every other stage: CAP
STAGE_ORDER = ["S0", "S1", "S2a", "S2b", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"]
CENTRELINE_STAGES = ("S0", "S1", "S2a", "S2b", "S3", "S4", "S5")  # no line raced: errors from the centreline

In [ ]:
stage_results = {"S0": None}  # TODO: S0 is Milestone 1's bar_results; then {stage: [five results]} for S1-S9
# TODO: and "S10": the candidate's five races, if you have a candidate (race(CANDIDATE, seed=s))
s10_submitted = None  # TODO: True or False (or None if there is no candidate)

ablation = []  # TODO: one dict per stage in STAGE_ORDER that you raced:
# {"stage", "idea", "clean" (of 5), "median", "spread", "time bought", "verdict", "max error"}
# TODO: print the table, one line per stage

fig, axes = None, None  # TODO: plt.subplots(2, 1, sharex=True): the times and medians above, the largest line error below

SUBMITTED_STAGE = None  # TODO: "S9" or "S10"
leave_one_out = []  # TODO: one dict per removal ("no line", "no grip", "no MPC", "no tune"):
# {"removed", "median", "cost", "verdict", "clean" (of 5)}
# TODO: print it

## Milestone 4

### Qualify

1. **Against the bar.** The submitted file's five races (its row of Milestone 3) against the bar's five: `beats_bar` is `True` when all ten are clean with 0 collisions and `noise_aware_better(the submitted file's times, bar_times)`; `bar_margin` is the bar's median minus the submitted file's (s).
2. **Against Part 1.** Race `agents/unit-07.py` (Part 1's file; it races the centreline and has no timing) at race seeds 0 to 4: `unit07_times`, `unit07_clean` (against `CENTRELINE`), and `beats_unit07` by Unit 1's rule.
3. **The submission check.** `report = check_submission(SUBMITTED, track="monza-like")`: the file exactly as you would submit it, the timing **on**, checked on its own track (Unit 13: a race-day file refuses another track's line).
   Print the report and `report.step_p99_ms` (your machine's, under its load: print it, do not compare it with a fixed number); `check_passed` is `report.ok`.
   Its race is on the check's own car, timed: its race time is not comparable with your table.
4. **The packaging checklist** (Unit 13, §6): in `checklist`, at least eight things you checked before submitting (the one file and its shape, Python 3.8, the values on their grid and written with `repr`, the header comment, the timing as shipped, the check on the race track and where to run it, the versions and the file's hash, the season's files and wording), each with what you found.

In [ ]:
submitted_times = None  # TODO: the five times of SUBMITTED_STAGE's row
beats_bar, bar_margin = None, None  # TODO

unit07_results = []  # TODO: race(UNIT_07, seed=s) for s in SEEDS
unit07_times, unit07_clean = None, None  # TODO
beats_unit07 = None  # TODO

report = None  # TODO: check_submission(SUBMITTED, track="monza-like")
# TODO: print(report) and report.step_p99_ms
check_passed = None  # TODO: report.ok

checklist = []  # TODO: at least eight strings, each a check and what you found

## Milestone 5

### The engineering report

Write the memo a race engineer would read before race day, in the answer cells below.
Every claim cites a number from your own tables (or from a given record, named as such); two to five sentences per answer.

1. **The table.** Paste your ablation table (the stage, its idea, clean, median, spread, time bought, Unit 1's verdict, max error), with the submitted file's row marked, and the seeds you raced.
2. **What each stage bought.** Where was most of the time bought, and how much of it before Unit 8?
   Give every "time bought" next to its spread: which gains are within the noise?
3. **The stage that cost time.** S1 is slower than the bar it builds on.
   Why, and what did it buy instead?
   What does S2a against S2b say about where a stage's value comes from?
4. **What race time does not show.** What did MPC (S7) and the respawn fallback (S8) buy in time, and what did they buy that a time-only ablation cannot show?
   Name the evidence: the line error, and Unit 13's record of pursuit on the validation cars.
5. **The order, and why stages do not add.** Compare each removal of the leave-one-out with the time the same idea bought in the build-up.
   Why do they differ, and why must a report state the order of its ablation?
6. **Your choice.** Your knob and why, its training walk (the edge, the margin), its validation count, your pre-registration and your test result (and the second knob's evidence, if you chose `TARGET_SPEED`).
   Which file do you submit, and why?
   If the candidate failed, state its nominal gain and its test failures as two separate facts: why can you not net them into one expected time?
7. **What next.** One thing you would change or measure next, the cars it would need (a fresh pre-registered set), and the result that would make you submit it.
8. **The CARLA caveat.** Every number above is racekit's.
   What does that mean for race day, and which target can only CARLA test (the book's final qualification: five official-runner launches, every one clean, median at most 329 s)?

**1.** _Your answer._

**2.** _Your answer._

**3.** _Your answer._

**4.** _Your answer._

**5.** _Your answer._

**6.** _Your answer._

**7.** _Your answer._

**8.** _Your answer._

## Make it yours

Choose one or more extensions after your five milestones are done.
None may change the submitted file: it was decided by its test.

- **The other order.** Run the build-up with the line before the grip (S5 and S6 swapped).
  What does each now buy, and what does that say about additive thinking?
- **A second choice, honestly.** Take a knob you did not tune through training and validation.
  Write the pre-registration you would need before a test, and name the fresh cars it would need (never 72 to 143).
- **MPC's margin, measured.** Race S6 and S7 of your ablation on five validation cars of your own choosing, and compare their clean counts and line errors with the nominal car's.
  (The validation cars may inform you here: they decide nothing.)
- **Profile the submitted file.** `cProfile` over one 1-lap race of the submitted file, timing off (Unit 12): where does `step()`'s time go, and how does `check_submission`'s p99 compare with the 5 ms budget?
- **On CARLA.** In the lab, race your submitted file on CARLA's Monza and put the result next to racekit's in your report.

## Requirements checklist

**Your capstone must: make the bar clean with its own knob; make and test one choice of your own by a rule written before the test; name the incumbent, the candidate and the submitted file; build the submitted file up from the bar on seeds 0 to 4 with spread and Unit 1's verdict; cross-check with the leave-one-out; qualify the submitted file; and write the engineering report.**

Before you hand in, check every box.

- [ ] Milestone 1: the shipped starter's capped race, why it is not clean on this car, the `TARGET_SPEED` grid, and the bar on five seeds.
- [ ] Milestone 2: the knob and why; the grid step written before any race; the training walk on whole ticks with its edge and margin; the validation count; the pre-registration written and run **before** the test cell; your test cars 120 to 143 raced **once**; the submitted file decided by the rule; a second knob's evidence if you chose `TARGET_SPEED`.
- [ ] Milestone 3: every stage on seeds 0 to 4 (or 0 to 2, said so), with clean, median, spread, time bought, Unit 1's verdict and max error; the plot; the candidate's row marked submitted or not; the leave-one-out.
- [ ] Milestone 4: the submitted file beats the bar by Unit 1's rule at 0 collisions; against `unit-07.py`; `check_submission` on `monza-like` with p99 printed; the packaging checklist.
- [ ] Milestone 5: all eight answers, each citing your numbers.
- [ ] I never raced cars 72 to 119, cars k ≥ 1000, `test-circuit-seed2027` or `test-circuit-seed2028`, and my test cars only once.
- [ ] Every race except `check_submission`'s ran with the timing off.
- [ ] The notebook runs top to bottom without errors (except the test cell, which has run its one time).